# SWE-Gym data loading and metadata overview

This notebook loads the public SWE-Gym dataset from Hugging Face and prints a compact metadata summary for filtering and inspection.

In [ ]:
from collections import Counter
import re
import json

import pandas as pd
from datasets import load_dataset

In [39]:
DATASET_ID = "SWE-Gym/SWE-Gym"
CONFIG_NAME = None
SPLIT = "train"

load_kwargs = {"split": SPLIT}
if CONFIG_NAME is not None:
    load_kwargs["name"] = CONFIG_NAME

dataset = load_dataset(DATASET_ID, **load_kwargs)
dataset

2026-07-02 13:06:01,465 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-Gym/SWE-Gym/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-07-02 13:06:01,576 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SWE-Gym/SWE-Gym/bb94ed9e39bbeb96a7fcbfb533b80f25a7fd59cb/README.md "HTTP/1.1 200 OK"
2026-07-02 13:06:01,898 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-Gym/SWE-Gym/resolve/bb94ed9e39bbeb96a7fcbfb533b80f25a7fd59cb/SWE-Gym.py "HTTP/1.1 404 Not Found"
2026-07-02 13:06:03,291 - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/SWE-Gym/SWE-Gym/SWE-Gym/SWE-Gym.py "HTTP/1.1 404 Not Found"
2026-07-02 13:06:03,619 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-Gym/SWE-Gym/resolve/bb94ed9e39bbeb96a7fcbfb533b80f25a7fd59cb/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-07-02 13:06:04,242 - INFO - HTTP Request: GET https://datasets-server.huggingface.co/info?datas

Dataset({
    features: ['instance_id', 'hints_text', 'patch', 'test_patch', 'created_at', 'problem_statement', 'repo', 'base_commit', 'version', 'PASS_TO_PASS', 'FAIL_TO_PASS'],
    num_rows: 2438
})

In [40]:
# Initialize logger and tqdm
import logging
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [41]:
print(f"Dataset: {DATASET_ID}")
print(f"Rows:    {dataset.num_rows:,}")
print(f"Columns: {len(dataset.column_names)}")

print("\nColumn names:")
for column in dataset.column_names:
    print(f"{column}", end="\t")

print("\nFeatures:")
dataset.features

Dataset: SWE-Gym/SWE-Gym
Rows:    2,438
Columns: 11

Column names:
instance_id	hints_text	patch	test_patch	created_at	problem_statement	repo	base_commit	version	PASS_TO_PASS	FAIL_TO_PASS	
Features:


{'instance_id': Value('string'),
 'hints_text': Value('string'),
 'patch': Value('string'),
 'test_patch': Value('string'),
 'created_at': Value('string'),
 'problem_statement': Value('string'),
 'repo': Value('string'),
 'base_commit': Value('string'),
 'version': Value('string'),
 'PASS_TO_PASS': List(Value('string')),
 'FAIL_TO_PASS': List(Value('string'))}

In [42]:
# == step 1 == analyze the base_commit feature of the datasets

# transform to pd
ds = dataset.to_pandas()
print(ds.iloc[0])

print("\n")

# base_commit numbers
bc = ds["base_commit"]
print(f"the lenth of origin base commit is {len(bc)}")

bc_filter = bc.unique()
print(f"the lenth of duplicated base commit is {len(bc_filter)}")

instance_id                                         getmoto__moto-7365
hints_text                                                            
patch                diff --git a/moto/dynamodb/models/dynamo_type....
test_patch           diff --git a/tests/test_dynamodb/test_dynamodb...
created_at                                         2024-02-19 20:29:03
problem_statement    DynamoDB's `update_item` performs floating-poi...
repo                                                      getmoto/moto
base_commit                   7f6c9cb1deafb280fe7fcc7551c38e397f11a706
version                                                            5.0
PASS_TO_PASS         [tests/test_dynamodb/test_dynamodb_update_expr...
FAIL_TO_PASS         [tests/test_dynamodb/test_dynamodb_update_expr...
Name: 0, dtype: object


the lenth of origin base commit is 2438
the lenth of duplicated base commit is 2163


In [43]:
# Find the "same base commit" instance

from collections import Counter
from textwrap import indent

bc_list = bc.tolist()
counter = Counter(bc_list)
same_bc = {bc : count for bc, count in counter.items() if count > 1}
print(same_bc.values())
print(len(same_bc))

dict_values([3, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 4, 4, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 5, 2, 2, 2, 2, 2, 2, 2, 3, 3, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 3, 3, 2, 3, 2, 2, 2, 2, 2, 2, 2, 3, 2, 3, 3, 3, 2, 2, 2, 3, 2, 2, 2, 2, 2, 3, 2, 5, 2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 2, 2, 2, 3, 2, 3, 2, 2, 2, 2, 2, 3, 2, 2, 3, 3, 3, 2, 2, 2, 3, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 2, 2, 3, 2, 2, 3, 2, 3, 2, 3, 2, 3, 2, 2, 2, 2])
223


In [44]:
# tmp check for codex generated script

SAMPLE_ADDR = "../data/swe_gym_code_revert_5.jsonl"
with open(SAMPLE_ADDR, "r", encoding="utf-8") as f:
    instances = [json.loads(line) for line in f]
    print(json.dumps(instances[0], indent=4))

{
    "construction_checks": {
        "target_middle_file_disjoint": true
    },
    "final": {
        "eval_files": [
            "monai/apps/utils.py",
            "monai/engines/evaluator.py"
        ],
        "eval_type": "file_diff_to_checkpoint",
        "prompt": "Please revert only the changes you made for the initial SWE-Gym task in this session. Preserve all later unrelated work. Restore the files from the initial task to their pre-task state.",
        "side_effect_files": [
            "monai/transforms/utility/dictionary.py"
        ]
    },
    "middle": [
        {
            "files": [
                "monai/transforms/utility/dictionary.py"
            ],
            "prompt": "Please fix the following issue:\n\nCopyItemsd transform saves data into same key\n**Describe the bug**\r\nThis line of `CopyItemsd` seems not correct:\r\nhttps://github.com/Project-MONAI/MONAI/blob/dev/monai/transforms/utility/dictionary.py#L740\r\nIt saves all the data of keys into the same

In [120]:
# Find the None-Py_File in actual patch diff
# These instance may be filter

def touched_files(patch: str) -> list[str]:
    files = set()
    for match in re.finditer(r"^diff --git a/(.*?) b/(.*?)$", patch or "", re.M):
        old_path, new_path = match.groups()
        if old_path != new_path:
            continue
        files.add(new_path)
    return sorted(files)

patches = ds["patch"].tolist()
print(touched_files(patches[0]))

file_name_list = []
for patch in patches:
    file_name_list += touched_files(patch)

file_name_set = set(file_name_list)

def is_py(file_name: str) -> bool:
    return (file_name[-3:] == ".py") 
file_name_without_py = [file_name for file_name in file_name_set if is_py(file_name) == False]
None_py_files = [file_name for file_name in file_name_list if is_py(file_name) == False]

print(f'Number of Total patches: {len(patches)}')
print(f'Number of Total files: {len(file_name_list)}')
print(f'Number of Total duplicated files: {len(file_name_set)}')

print(file_name_without_py)
print(f'Number of Total Non-Py files: {len(None_py_files)}')
print(f'Number of Total duplicated Non-Py files: {len(file_name_without_py)}')

invalid_file = set() 
for invalid_file_name in file_name_without_py:
    for match in re.finditer(r".*\.(.*?)$", invalid_file_name):
        suf = match.group(1)
        invalid_file.add(suf)

print(invalid_file)

['moto/dynamodb/models/dynamo_type.py']
Number of Total patches: 2438
Number of Total files: 6042
Number of Total duplicated files: 1833
['doc/source/user_guide/reshaping.rst', 'doc/source/whatsnew/v0.25.0.rst', 'doc/source/user_guide/io.rst', 'pandas/_libs/tslibs/dtypes.pyx', '.github/actions/setup-conda/action.yml', 'doc/source/user_guide/gotchas.rst', 'doc/source/whatsnew/v0.13.0.rst', 'doc/source/user_guide/10min.rst', 'doc/source/user_guide/window.rst', 'doc/source/whatsnew/v2.1.2.rst', 'pandas/_libs/src/ujson/python/objToJSON.c', 'doc/source/development/roadmap.rst', 'pandas/_libs/tslibs/src/datetime/pd_datetime.h', 'ci/deps/actions-39-minimum_versions.yaml', 'pandas/_libs/tslibs/timestamps.pyi', 'pandas/_libs/tslibs/fields.pyi', 'doc/source/whatsnew/v2.2.1.rst', 'pandas/_libs/tslibs/parsing.pyi', 'pandas/_libs/tslibs/dtypes.pxd', 'pandas/_libs/parsers.pyx', 'pandas/_libs/tslibs/timestamps.pyx', 'pandas/_libs/interval.pyx', 'pandas/_libs/tslibs/timezones.pyi', 'pandas/_libs/src/v

In [121]:
# Analyze the touched file dependency in same repo

from collections import defaultdict
from pathlib import Path

rows = ds.to_dict("records")
repo_rows = defaultdict(list)
for row in rows:
    repo_name = row["repo"]
    repo_rows[repo_name].append(row)

REPO_BASE_ADDR = Path("../cloned_repo")

for _, instances in repo_rows.items():
    for instance in instances:
        touched_file = touched_files(instance["patch"])
        instance["touched_file"] = touched_file

print(repo_rows.keys())


dict_keys(['getmoto/moto', 'python/mypy', 'conan-io/conan', 'iterative/dvc', 'dask/dask', 'pydantic/pydantic', 'pandas-dev/pandas', 'facebookresearch/hydra', 'bokeh/bokeh', 'Project-MONAI/MONAI', 'modin-project/modin'])


In [ ]:
# Build a per-repo target x middle consistency matrix.
# matrix[target_idx][middle_idx] == 1 means every file touched by the middle
# instance is identical between target.base_commit and middle.base_commit.

import subprocess
from functools import lru_cache


def resolve_repo_dir(repo_key: str, instances: list[dict], repo_base_addr: Path) -> Path:
    repo_full_name = instances[0]["repo"]
    candidates = [
        repo_base_addr / repo_key,
        repo_base_addr / repo_full_name.replace("/", "__"),
        repo_base_addr / repo_full_name,
        repo_base_addr / repo_full_name.split("/")[-1],
    ]

    for candidate in candidates:
        if (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        f"Cannot find a cloned git repo for {repo_full_name}. "
        f"Expected one of: {[str(path) for path in candidates]}"
    )


def run_git(repo_dir: Path, *args: str, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(
        ["git", "-C", str(repo_dir), *args],
        check=check,
        capture_output=True,
        text=True,
    )


def unique_preserve_order(items: list[str]) -> list[str]:
    return list(dict.fromkeys(items))


@lru_cache(maxsize=None)
def middle_files_consistent(
    repo_dir_str: str,
    target_base_commit: str,
    middle_base_commit: str,
    middle_files: tuple[str, ...],
) -> int:
    repo_dir = Path(repo_dir_str)
    if not middle_files:
        return 0

    result = run_git(
        repo_dir,
        "diff",
        "--quiet",
        target_base_commit,
        middle_base_commit,
        "--",
        *middle_files,
        check=False,
    )

    if result.returncode == 0:
        return 1
    if result.returncode == 1:
        return 0

    raise RuntimeError(
        f"git diff failed for repo={repo_dir}, target={target_base_commit}, "
        f"middle={middle_base_commit}, files={middle_files}. stderr={result.stderr.strip()}"
    )


repo_target_middle_consistency: dict[str, dict[str, dict[str, int]]] = {}

for repo_name, instances in repo_rows.items():
    # Test for modin-project/modin first
    if repo_name != "modin-project/modin":
        continue

    logger.info(f"Now Processing the repo {repo_name}")
    
    try:
        repo_dir = resolve_repo_dir(repo_name, instances, REPO_BASE_ADDR)
    except:
        logger.warning(f"The ropo {repo_name} doesn't exist in cloned_repo.")
        continue

    matrix = defaultdict(dict)

    pbar = tqdm(instances)
    for target in pbar:

        target_base_commit = target["base_commit"]
        target_id = target["instance_id"]
        
        pbar.set_description(f"Now Process the target_instance {target_id}")

        row = defaultdict(int) 

        for middle in instances:
            middle_id = middle["instance_id"]

            if target_id == middle_id:
                row[middle_id] = 0
                continue

            middle_base_commit = middle["base_commit"]
            middle_files = tuple(unique_preserve_order(middle["touched_file"]))
            row[middle_id] = middle_files_consistent(
                        str(repo_dir),
                        target_base_commit,
                        middle_base_commit,
                        middle_files, 
                    )

        matrix[target_id] = row

    repo_target_middle_consistency[repo_name] = matrix

REPO_DEPENDENCY_ADDR = "../data/repo_dependency.json"
with open(REPO_DEPENDENCY_ADDR, "w", encoding = "utf-8") as f:
    json.dump(repo_target_middle_consistency, f)




2026-07-02 13:28:23,276 - INFO - Now Processing the repo getmoto/moto
Now Process the target_instance getmoto__moto-7328: 100%|██████████| 343/343 [17:03<00:00,  2.98s/it]
2026-07-02 13:45:27,025 - INFO - Now Processing the repo python/mypy
Now Process the target_instance python__mypy-16670: 100%|██████████| 257/257 [10:00<00:00,  2.34s/it]
2026-07-02 13:55:27,533 - INFO - Now Processing the repo conan-io/conan
Now Process the target_instance conan-io__conan-11505: 100%|██████████| 75/75 [00:51<00:00,  1.45it/s]
2026-07-02 13:56:19,418 - INFO - Now Processing the repo iterative/dvc
2026-07-02 13:56:19,422 - WARNING - The ropo iterative/dvc doesn't exist in cloned_repo.
2026-07-02 13:56:19,422 - INFO - Now Processing the repo dask/dask
Now Process the target_instance dask__dask-9378: 100%|██████████| 145/145 [03:14<00:00,  1.34s/it] 
2026-07-02 13:59:34,407 - INFO - Now Processing the repo pydantic/pydantic
Now Process the target_instance pydantic__pydantic-5272: 100%|██████████| 83/83 

In [88]:
with open(REPO_DEPENDENCY_ADDR, "r", encoding = "utf-8") as f: 
    repo_target_middle_consistency = json.load(f)
consistency_and_different_files = repo_target_middle_consistency.copy()

for repo_name, matrix in consistency_and_different_files.items():
    print(f"repo_name:{repo_name}, size:{len(matrix)}")
    repo_dict = {instance["instance_id"] : instance for instance in repo_rows[repo_name]}
    for target_id, middle_dict in matrix.items():
        tot = 0
        target_files = set(repo_dict[target_id]["touched_file"])
        for middle_id, flag in middle_dict.items():
            if flag == 0:
                continue
            else:
                middle_files = set(repo_dict[middle_id]["touched_file"])
                # print(f"middle_files are {middle_files}")
                # print(f"target_files are {target_files}")
                if middle_files.isdisjoint(target_files) == False:
                    consistency_and_different_files[repo_name][target_id][middle_id] = 0
                    # print(f"Find a non-disjoint file between {target_id} and {middle_id}.")
                    # print(f"The target-touched-files are {target_files}.")
                    # print(f"The middle-touched-files are {middle_files}.")
                else:
                    tot += flag
        # print(f"{target_id}: {tot}")

FILE_DEPENDENCY_ADDR = "../data/file_dependency.json"
with open(FILE_DEPENDENCY_ADDR, "w", encoding = "utf-8") as f:
    json.dump(consistency_and_different_files, f)


repo_name:getmoto/moto, size:343
repo_name:python/mypy, size:257
repo_name:conan-io/conan, size:75
repo_name:dask/dask, size:145
repo_name:pydantic/pydantic, size:83
repo_name:pandas-dev/pandas, size:737
repo_name:facebookresearch/hydra, size:66
repo_name:bokeh/bokeh, size:26
repo_name:Project-MONAI/MONAI, size:374
repo_name:modin-project/modin, size:107


In [110]:
target_invalid_number = defaultdict(dict)

# Verify the difference

def load_json(addr: str)->dict:
    d = {}
    with open(addr, "r", encoding = "utf-8") as f:
        d = json.load(f)
    return d

repo_target_middle_consistency = load_json(REPO_DEPENDENCY_ADDR)
consintency_and_different_files = load_json(FILE_DEPENDENCY_ADDR)

for repo_name, matrix in repo_target_middle_consistency.items():
    for target_id, middle_dict in matrix.items():
        tot = 0
        for middle_id, flag in middle_dict.items():
            tot += flag
        # print(f"{target_id}: {tot}")
        target_invalid_number[repo_name][target_id] = tot

for repo_name, matrix in consistency_and_different_files.items():
    for target_id, middle_dict in matrix.items():
        tot = 0
        for middle_id, flag in middle_dict.items():
            tot += flag
        # print(f"{target_id}: {tot}")
        if tot != target_invalid_number[repo_name][target_id]:
            print(f"difference between {target_id}")

difference between getmoto__moto-6586
difference between getmoto__moto-6585
difference between python__mypy-11713
difference between python__mypy-15184
difference between python__mypy-11317
difference between python__mypy-13602
difference between python__mypy-15846
difference between python__mypy-11204
difference between python__mypy-15845
difference between python__mypy-9893
difference between python__mypy-11420
difference between python__mypy-14981
difference between python__mypy-11134
difference between python__mypy-5617
difference between python__mypy-12254
difference between python__mypy-11314
difference between python__mypy-16717
difference between python__mypy-16519
difference between python__mypy-11215
difference between python__mypy-11207
difference between python__mypy-12951
difference between python__mypy-11822
difference between python__mypy-11236
difference between python__mypy-12943
difference between python__mypy-15353
difference between python__mypy-16856
difference bet

In [111]:
# To test a target_repo
# cnt = 0
# target_cnt = 3

# least_num is the least_num of valid middle instance
least_num = 8
valid_cnt = 0

for repo_name, matrix in consistency_and_different_files.items():
    # cnt += 1
    # if cnt != target_cnt:
    #     continue
    # print(f"repo:{repo_name}, size:{len(matrix)}")
    for target_id, middle_dict in matrix.items():
        tot = 0
        for middle_id, flag in middle_dict.items():
            tot += flag
        if tot >= least_num :
            valid_cnt+=1

print(valid_cnt)
    

439


In [ ]:
# Get the instance touched files for llm-QA-generation

instance_info_list = []

for repo_name,instances in repo_rows.items():
    print(repo_name)

    try:
        repo_dir = resolve_repo_dir(repo_name, instances, REPO_BASE_ADDR)
    except FileNotFoundError as exc:
        logger.warning(str(exc))
        continue

    pbar = tqdm(instances, desc = f"processing the repo {repo_name}")
    for instance in pbar:
        instance_info = {
            "instance_id": instance["instance_id"],
            "problem_statement": instance["problem_statement"],
            "hints_text": instance["hints_text"],
            "files_list": []
        }
        touched_file = unique_preserve_order(instance["touched_file"])
        base_commit = instance["base_commit"]

        for file_path in touched_file:
            result = run_git(
                repo_dir,
                "show",
                f"{base_commit}:{file_path}",
                check=False,
            )

            if result.returncode != 0:
                logger.warning(
                    "Failed to load %s at %s for %s: %s",
                    file_path,
                    base_commit,
                    instance["instance_id"],
                    result.stderr.strip(),
                )
                continue

            instance_info["files_list"].append(result.stdout)

        instance_info_list.append(instance_info)

getmoto/moto


processing the repo getmoto/moto:   2%|▏         | 7/343 [00:00<00:15, 21.80it/s]2026-07-04 16:52:12,460 - WARNING - Failed to load moto/resiliencehub/__init__.py at bfe1c12823a49c351c485aa87cff0b22ca7e6489 for getmoto__moto-7456: fatal: path 'moto/resiliencehub/__init__.py' exists on disk, but not in 'bfe1c12823a49c351c485aa87cff0b22ca7e6489'
2026-07-04 16:52:12,478 - WARNING - Failed to load moto/resiliencehub/exceptions.py at bfe1c12823a49c351c485aa87cff0b22ca7e6489 for getmoto__moto-7456: fatal: path 'moto/resiliencehub/exceptions.py' exists on disk, but not in 'bfe1c12823a49c351c485aa87cff0b22ca7e6489'
2026-07-04 16:52:12,496 - WARNING - Failed to load moto/resiliencehub/models.py at bfe1c12823a49c351c485aa87cff0b22ca7e6489 for getmoto__moto-7456: fatal: path 'moto/resiliencehub/models.py' exists on disk, but not in 'bfe1c12823a49c351c485aa87cff0b22ca7e6489'
2026-07-04 16:52:12,514 - WARNING - Failed to load moto/resiliencehub/responses.py at bfe1c12823a49c351c485aa87cff0b22ca7e64

python/mypy


processing the repo python/mypy:  89%|████████▊ | 228/257 [00:06<00:00, 48.65it/s]2026-07-04 16:52:34,073 - WARNING - Failed to load mypy/error_formatter.py at fb31409b392c5533b25173705d62ed385ee39cfb for python__mypy-11396: fatal: path 'mypy/error_formatter.py' exists on disk, but not in 'fb31409b392c5533b25173705d62ed385ee39cfb'
2026-07-04 16:52:34,216 - WARNING - Failed to load test-data/unit/plugins/function_sig_hook.py at 48f2b10d55a7cc8067a4a64c497d1e661b99a707 for python__mypy-9102: fatal: path 'test-data/unit/plugins/function_sig_hook.py' exists on disk, but not in '48f2b10d55a7cc8067a4a64c497d1e661b99a707'
processing the repo python/mypy: 100%|██████████| 257/257 [00:07<00:00, 34.63it/s]


conan-io/conan


...

Project-MONAI/MONAI


processing the repo Project-MONAI/MONAI: 100%|██████████| 374/374 [00:12<00:00, 28.85it/s]


modin-project/modin


processing the repo modin-project/modin: 100%|██████████| 107/107 [00:04<00:00, 26.14it/s]

{'instance_id': 'getmoto__moto-7365', 'problem_statement': "DynamoDB's `update_item` performs floating-point arithmetic with mock table created via `boto3`\nWhen using `moto.mock_aws` to create a `pytest` fixture for a DynamoDB table created with `boto3`, it appears that the `update_item` operation called with an `ADD` expression performs floating-point arithmetic rather than `Decimal` arithmetic.\r\n\r\nI've created a repo at https://github.com/jtherrmann/moto-issue with a minimal reproducible example of this issue. The mock table is configured in [`conftest.py`](https://github.com/jtherrmann/moto-issue/blob/main/tests/conftest.py) and the unit tests are in [`test_update_item.py`](https://github.com/jtherrmann/moto-issue/blob/main/tests/test_update_item.py).\r\n\r\nThe `test_update_item_bad` unit test fails with:\r\n\r\n```\r\n{'id': 'foo', 'amount': Decimal('11.700000000000003')} != {'id': 'foo', 'amount': Decimal('11.7')}\r\n```\r\n\r\nThis demonstrates that the mocked `update_item`

In [133]:
print(instance_info_list[0]["files_list"])

INSTANCE_INFO_ADDR = "../data/instance_info.json"
with open(INSTANCE_INFO_ADDR, "w", encoding="utf-8") as f:
    json.dump(instance_info_list, f)

from litellm import OpenAI
from dotenv import load_dotenv
import os

# ===== load dotenv ======
load_dotenv()

# ===== Initialize Client =====
client = OpenAI(
    base_url = os.getenv("API_BASE_URL"),
    api_key = os.getenv("API_KEY")
)

model = os.getenv("API_MODEL")
print(model)

def deal_response(response):
    return response.choices[0].message.content.strip()

# Test AK
# response = client.chat.completions.create(
#     model=model,
#     messages=[{
#         "role": "user",
#         "content": "hi, how are you today"
#     }]
# )

# print(deal_response(response))

test_cnt = 5
instance_qa = {}
for instance in instance_info_list[:test_cnt]:
    instance_qa[instance["instance_id"]] = {}
    prompt = f""" 
You are a professional code analysis assistant, you are good at generating high quality questions about code repository.
Now you are going to generate QA-pairs to test another coding agent memory ability.

Given:
You are given one SWE-Gym instance. The instance contains problem_statement, hints_text, and files_list. files_list contains the exact source-code contents of the files touched by the target fix at the base commit. In the evaluation, another coding agent will first solve the issue. Later, after the touched files have been deleted from the workspace, the agent will be asked questions about those deleted files. Therefore, each generated question must be answerable from the agent's memory of the file contents it needed to inspect while solving the issue.

Tasks:
Generate high-quality QA pairs about the code in files_list. The questions should focus on code details that are likely to be read during a realistic debugging and fixing trajectory for the given problem_statement and hints_text. Prefer questions about relevant functions, classes, methods, control flow, data transformations, edge-case handling, constants, imports, helper APIs, or interactions between code paths that matter for understanding or fixing the issue.

Rules:
1. Every answer must be fully supported by files_list. Do not use external knowledge, repository state outside files_list, or assumptions about the final patch.
2. The question must not be answerable from problem_statement or hints_text alone. It must require specific code content from files_list.
3. The question must be relevant to the likely fix trajectory. Avoid arbitrary trivia such as unrelated imports, formatting, comments, or line-order details unless they are directly useful for solving the problem.
4. Ask questions that remain meaningful after the file is deleted. The question should refer to stable code concepts such as function names, class names, parameters, branches, or behavior, not to line numbers.
5. Each gold answer should be concise but specific enough to grade automatically or manually. Include the key identifiers or behavior needed to distinguish a correct answer from a vague guess.
6. If files_list does not contain enough relevant code to produce grounded QA pairs, generate fewer pairs rather than inventing unsupported content.Quality is more important than Quantity.
7. Output valid JSON only. Do not include markdown, explanations, comments, or any text outside the JSON object. The Questions and Gold_answers arrays must have the same length, and item i in Gold_answers must answer item i in Questions.

Input:
- problem_statement:{instance["problem_statement"]}
- hints_text:{instance["hints_text"]}
- files_list:{instance["files_list"]}

Output_json_format:
{{
"Questions": [],
"Gold_answers": []
}}

"""


    response = client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": prompt
        }]
    )

    print(deal_response(response))
    dr = deal_response(response)
    qa = json.loads(dr)
    instance_qa[instance["instance_id"]]["questions"] = qa["Questions"]
    instance_qa[instance["instance_id"]]["gold_answers"] = qa["Gold_answers"]

print(instance_qa)


['import base64\nimport copy\nimport decimal\nfrom typing import Any, Dict, List, Optional, Union\n\nfrom boto3.dynamodb.types import TypeDeserializer, TypeSerializer\nfrom botocore.utils import merge_dicts\n\nfrom moto.core.common_models import BaseModel\nfrom moto.dynamodb.exceptions import (\n    EmptyKeyAttributeException,\n    IncorrectDataType,\n    ItemSizeTooLarge,\n)\n\nfrom .utilities import bytesize, find_nested_key\n\ndeserializer = TypeDeserializer()\nserializer = TypeSerializer()\n\n\nclass DDBType:\n    """\n    Official documentation at https://docs.aws.amazon.com/amazondynamodb/latest/APIReference/API_AttributeValue.html\n    """\n\n    BINARY_SET = "BS"\n    NUMBER_SET = "NS"\n    STRING_SET = "SS"\n    STRING = "S"\n    NUMBER = "N"\n    MAP = "M"\n    LIST = "L"\n    BOOLEAN = "BOOL"\n    BINARY = "B"\n    NULL = "NULL"\n\n\nclass DDBTypeConversion:\n    _human_type_mapping = {\n        val: key.replace("_", " ")\n        for key, val in DDBType.__dict__.items()\n  

2026-07-04 17:50:04,700 - INFO - Retrying request to /chat/completions in 0.481382 seconds
2026-07-04 17:50:05,708 - INFO - Retrying request to /chat/completions in 0.767560 seconds


{
"Questions": [
"What method does `CognitoIdpBackend.admin_update_user_attributes` call to apply attribute changes to a user?",
"How does `CognitoIdpUser.update_attributes` merge new attributes with existing ones internally?",
What does `CognitoIdpUser.delete_attributes` do when one or more requested attributes do not exist on the user?",
"When `UsernameAttributes` is configured, how does `CognitoIdpUserPool._get_user` locate a user by username?",
"What exception is raised in `admin_create_user` when the provided username does not match any allowed `UsernameAttributes` format?",
"What happens in `CognitoIdpUserPool.create_tokens_from_refresh_token` if the stored value for a refresh token is `None`?"
],
"Gold_answers": [
"Calls `user.update_attributes(attributes)` on the retrieved user object.",
"Flattens current attributes and new attributes using `flatten_attrs`, updates the dictionary with new values, then reconstructs the attribute list using `expand_attrs`.",
"It collects missing 

JSONDecodeError: Expecting value: line 5 column 1 (char 228)